In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


BRONZE_SCHEMA = "football_data_platform.bronze"
SILVER_SCHEMA = "football_data_platform.silver"
SILVER_TABLE = f"{SILVER_SCHEMA}.venues_data"


def clean_strings(df: DataFrame) -> DataFrame:
    """Trim string columns and convert empty strings to NULL."""

    for column, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(
                column,
                F.when(
                    F.trim(F.col(f"`{column}`")) == "",
                    None
                ).otherwise(
                    F.trim(F.col(f"`{column}`"))
                )
            )

    return df


# ---------------------------------------------------------
# 1. Read Bronze tables
# ---------------------------------------------------------
table_name = f"{BRONZE_SCHEMA}.api_venues_data"

if not spark.catalog.tableExists(table_name):
    print(f"WARNING: Table does not exist: {table_name}")

df = spark.table(table_name)
df = clean_strings(df)


# ---------------------------------------------------------
# 3. Select and standardize columns
# ---------------------------------------------------------

df_silver = df.select(
    # Business columns - venue information
    F.col("address").cast("string").alias("address"),
    F.col("capacity").cast("int").alias("capacity"),
    F.col("city").cast("string").alias("city"),
    F.col("country").cast("string").alias("country"),
    F.col("id").cast("string").alias("venue_id"),
    F.col("name").cast("string").alias("name"),
    F.col("surface").cast("string").alias("surface"),

    # Bronze metadata
    F.col("_ingested_at").cast("timestamp").alias("_bronze_ingested_at"),
    F.col("_source_file").cast("string").alias("_bronze_source_file"),
    F.col("_source_system").cast("string").alias("_source_system"),
)


# ---------------------------------------------------------
# 4. Remove duplicates
# ---------------------------------------------------------

duplicate_columns = [
    "venue_id"
]

duplicate_count = (
    df_silver
    .groupBy(duplicate_columns)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate groups found: {duplicate_count}")

if duplicate_count > 0:
    df_silver = df_silver.dropDuplicates(duplicate_columns)
    print("Duplicates removed.")
else:
    print("No duplicates found. No changes made.")


# ---------------------------------------------------------
# 5. Add Silver metadata
# ---------------------------------------------------------

df_silver = df_silver.withColumn(
    "_silver_processed_at",
    F.current_timestamp()
)


# ---------------------------------------------------------
# 6. Basic validation
# ---------------------------------------------------------

null_venue_names = df_silver.filter(
    F.col("name").isNull()
).count()

if null_venue_names > 0:
    print(f"WARNING: NULL venue_name: {null_venue_names}")

print(f"Silver rows: {df_silver.count()}")


# ---------------------------------------------------------
# 7. Write Silver
# ---------------------------------------------------------

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Created {SILVER_TABLE}")